In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_excel("datasets/sample_-_superstore.xls")

In [6]:
analysis_date = pd.Timestamp("2026-10-24")

def calculate_baseline(data):
    total_sales = data["Sales"].sum()
    total_profit = data["Profit"].sum()

    return {
        "Rows": len(data),
        "Total Sales": total_sales,
        "Total Profit": total_profit,
        "Profit Margin (%)": (
            total_profit / total_sales * 100
            if total_sales != 0 else None
        ),
        "Total Orders": data["Order ID"].nunique(),
        "Total Quantity": data["Quantity"].sum()
    }

# Full dataset
full_results = calculate_baseline(df)

# Orders up to the analysis date
historical_df = df[df["Order Date"] <= analysis_date]
historical_results = calculate_baseline(historical_df)

comparison = pd.DataFrame({
    "Full Dataset": full_results,
    "As of 2026-10-24": historical_results
})

print(comparison.round(2))

                   Full Dataset  As of 2026-10-24
Rows                   10194.00           9195.00
Total Sales          2326534.35        2115813.28
Total Profit          292296.81         273117.69
Profit Margin (%)         12.56             12.91
Total Orders            5111.00           4588.00
Total Quantity         38654.00          34787.00


In [7]:
# Analyze profitability across discount levels

discount_analysis = (
    historical_df
    .groupby("Discount")
    .agg(
        Orders=("Order ID", "nunique"),
        Rows=("Order ID", "size"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

discount_analysis["Profit Margin (%)"] = (
    discount_analysis["Profit"]
    / discount_analysis["Sales"]
    * 100
)

discount_analysis = discount_analysis.sort_values("Discount")

discount_analysis.round(2)

,Discount,Orders,Rows,Sales,Profit,Quantity,Profit Margin (%)
0,0.00,2433,4443,999084.60,296383.55,16913,29.67
1,0.10,86,91,51190.48,8451.75,353,16.51
2,0.15,46,47,25037.44,1407.58,173,5.62
3,0.20,2193,3336,712341.21,84652.96,12453,11.88
4,0.30,192,208,96218.60,-9754.22,782,-10.14
5,0.32,24,24,13880.87,-2337.82,98,-16.84
6,0.40,165,185,106128.86,-20744.63,689,-19.55
7,0.45,9,10,4763.10,-2073.11,39,-43.52
8,0.50,59,61,49747.94,-16120.13,228,-32.40
9,0.60,123,132,6128.19,-5212.27,501,-85.05


In [8]:
category_discount_analysis = (
    historical_df
    .groupby(["Category", "Discount"])
    .agg(
        Rows=("Order ID", "size"),
        Orders=("Order ID", "nunique"),
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Quantity=("Quantity", "sum")
    )
    .reset_index()
)

category_discount_analysis["Profit Margin (%)"] = (
    category_discount_analysis["Profit"]
    / category_discount_analysis["Sales"]
    * 100
)

category_discount_analysis = category_discount_analysis.sort_values(
    ["Category", "Discount"]
)

category_discount_analysis.round(2)

,Category,Discount,Rows,Orders,Sales,Profit,Quantity,Profit Margin (%)
0,Furniture,0.00,785,686,233690.17,53738.95,3085,23.00
1,Furniture,0.10,73,68,43455.38,6533.59,293,15.04
2,Furniture,0.15,47,46,25037.44,1407.58,173,5.62
3,Furniture,0.20,568,504,201647.81,6317.22,2071,3.13
4,Furniture,0.30,203,188,92462.29,-10080.25,770,-10.90
5,Furniture,0.32,24,24,13880.87,-2337.82,98,-16.84
6,Furniture,0.40,67,64,39438.80,-14132.41,253,-35.83
7,Furniture,0.45,10,9,4763.10,-2073.11,39,-43.52
8,Furniture,0.50,52,51,20581.84,-12604.67,204,-61.24
9,Furniture,0.60,130,121,6074.23,-5182.00,492,-85.31
